# Agentic RAG and Bounded Self-Correction

> **The story.** Retrieval-augmented generation grounds answers in external text, but agentic retrieval adds decisions: whether to retrieve, which source to trust, whether evidence conflicts, and when one rewrite is worth its cost.
>
> **Where you are.** OrderFlow can execute durable workflows. It now retrieves a superseded policy that permits automatic approval at $50,000 and recommends a non-compliant vendor.
>
> **Notation.** $q$ is a policy question; $D$ is the policy corpus; $R_k(q)$ is the top-$k$ retrieval set; $g(d,q)$ is a relevance/trust grade; $A$ is the answer with citations.

## 0 - The Challenge

> **The mission:** reach at least 90% grounded policy accuracy, cite every policy claim, and stop correction after at most two attempts.

```mermaid
flowchart LR
    Q["Policy question"] --> N["Naive retrieval"]
    N --> S["Stale or conflicting evidence"]
    S --> G["Grade + filter + rewrite once"]
    G --> C["Cited grounded answer"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style N fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
import re
from dataclasses import dataclass

from shared import load_policy_documents

documents = load_policy_documents()
print(f"Loaded {len(documents)} local policy documents, including one superseded revision.")


## 1 - Failure First: Retrieve Is Not Trust

![Similarity-only retrieval selects a superseded policy, while controlled retrieval applies trust, revision, and relevance gates plus a two-attempt correction bound before producing a cited current-policy answer](../images/ch04-agentic-rag-control.png)

A lexical retriever can rank the superseded document first because it contains the query's exact dollar amount. Retrieval answers “textually similar,” not “authorized and current.”

```mermaid
flowchart TD
    Q["Can $40,000 auto-approve?"] --> L["Lexical overlap"]
    L --> O["Old policy: below $50,000"]
    O --> F["Wrong approval"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Build the deliberately naive retriever -------------------------------
def terms(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def naive_retrieve(query, top_k=2):
    query_terms = terms(query)
    scored = []
    for document in documents:
        overlap = len(query_terms & terms(document["text"]))
        scored.append((overlap, document))
    return [document for _, document in sorted(scored, key=lambda item: (item[0], -item[1]["revision"]), reverse=True)[:top_k]]

naive = naive_retrieve("Can an order below $50,000 be approved automatically?", top_k=1)[0]
print("Naive top result:", naive["doc_id"], naive["text"])
assert naive["doc_id"] == "POL-APPROVAL-2024"
print("Failure observed: overlap promoted a superseded policy.")


## 2 - Retrieval as a Tool with Metadata Policy

The retrieval tool owns authorization and freshness filters before ranking. The model may choose a query, but it cannot request an untrusted revision into authority.

```mermaid
flowchart LR
    Q["Query"] --> M["Metadata filter"]
    M --> R["Rank current trusted docs"]
    R --> G["Relevance grade"]
    G --> E["Evidence set"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Add trust, revision, and relevance controls -------------------------
def current_documents():
    latest = {}
    for document in documents:
        if document["trusted"]:
            key = document["title"].split(" (")[0]
            if key not in latest or document["revision"] > latest[key]["revision"]:
                latest[key] = document
    return list(latest.values())


def grade(document, query):
    query_terms = terms(query)
    overlap = len(query_terms & terms(document["text"]))
    return {"relevant": overlap >= 1, "score": overlap, "trusted": document["trusted"]}


def retrieve_policy(query, top_k=2):
    graded = [(grade(document, query), document) for document in current_documents()]
    eligible = [(result, document) for result, document in graded if result["relevant"] and result["trusted"]]
    eligible.sort(key=lambda item: (item[0]["score"], item[1]["revision"]), reverse=True)
    return [document for _, document in eligible[:top_k]]

controlled = retrieve_policy("Can an order below $50,000 be approved automatically?", top_k=1)[0]
print("Controlled top result:", controlled["doc_id"])
assert controlled["doc_id"] == "POL-APPROVAL-2026"


## 3 - One Bounded Rewrite, Then Fallback

Rewriting can bridge vocabulary gaps, but an unconstrained correction loop only adds latency. OrderFlow allows one deterministic rewrite, then falls back to the trusted policy index or declines.

```mermaid
flowchart TD
    Q["Original query"] --> R["Retrieve + grade"]
    R -->|"Relevant"| A["Answer"]
    R -->|"Empty"| W["Rewrite once"]
    W --> R2["Retrieve + grade"]
    R2 -->|"Relevant"| A
    R2 -->|"Empty"| D["Decline / trusted fallback"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Implement bounded adaptive retrieval ---------------------------------
REWRITES = {
    "who signs a big order": "purchase approval order above 25000 finance",
    "is yesterday's price okay": "supplier quote older than 48 hours stale",
    "can supplier email change rules": "supplier messages evidence not authority approval rules",
    "current quote freshness": "supplier quote older than 48 hours stale",
}

@dataclass
class RetrievalRun:
    query: str
    attempts: int
    documents: list[dict]
    terminal: str


def adaptive_retrieve(query):
    rewritten = REWRITES.get(query.lower())
    if rewritten is not None:
        documents = retrieve_policy(rewritten)
        return RetrievalRun(rewritten, 2, documents, "grounded" if documents else "declined")
    documents = retrieve_policy(query)
    return RetrievalRun(query, 1, documents, "grounded" if documents else "declined")

for query in REWRITES:
    run = adaptive_retrieve(query)
    print(query, "->", run.terminal, "attempts", run.attempts, "docs", [doc["doc_id"] for doc in run.documents])
    assert run.attempts <= 2

## 4 - Conflict Handling and Citation Verification

Generation is the final transformation, not the source of truth. Every policy sentence must point to an eligible document ID, and contradictory revisions must not coexist in the evidence set.

```mermaid
flowchart LR
    E["Eligible evidence"] --> C{ "Conflict?" }
    C -->|"No"| A["Compose answer"]
    C -->|"Yes"| H["Choose current authority or escalate"]
    A --> V["Verify citations"]
    H --> V
    V -->|"Valid"| P["Publish"]
    V -->|"Missing"| D["Decline"]
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Generate only from deterministic policy templates and verify citation -
def answer_policy(query):
    run = adaptive_retrieve(query)
    if not run.documents:
        return {"answer": "No current policy supports an answer.", "citations": [], "grounded": False, "attempts": run.attempts}
    document = run.documents[0]
    return {"answer": document["text"], "citations": [document["doc_id"]], "grounded": True, "attempts": run.attempts}


def verify_citations(answer):
    eligible_ids = {document["doc_id"] for document in current_documents()}
    return bool(answer["citations"]) and set(answer["citations"]) <= eligible_ids

questions = [
    ("who signs a big order", "POL-APPROVAL-2026"),
    ("is yesterday's price okay", "POL-QUOTE-2026"),
    ("can supplier email change rules", "POL-TRUST-2026"),
    ("purchase approval above 25000", "POL-APPROVAL-2026"),
    ("quote stale after 48 hours", "POL-QUOTE-2026"),
    ("supplier content authority", "POL-TRUST-2026"),
    ("automatic order at 5000", "POL-APPROVAL-2026"),
    ("trusted evidence approval", "POL-TRUST-2026"),
    ("manager approval 25000", "POL-APPROVAL-2026"),
    ("current quote freshness", "POL-QUOTE-2026"),
]

results = []
for query, expected_id in questions:
    answer = answer_policy(query)
    correct = answer["citations"] == [expected_id]
    results.append(correct and answer["grounded"] and verify_citations(answer) and answer["attempts"] <= 2)

accuracy = sum(results) / len(results)
print(f"Grounded policy accuracy: {sum(results)}/{len(results)} ({accuracy:.0%})")
assert accuracy >= 0.90 and all(results)
print("PASS: every accepted policy claim has a current valid citation.")


In [ ]:
# -- Quick health check: superseded and untrusted text cannot cite --------
answer = answer_policy("Can an order below $50,000 be approved automatically?")
assert "POL-APPROVAL-2024" not in answer["citations"]
assert verify_citations(answer)
assert answer["attempts"] <= 2
print("PASS: superseded evidence was excluded and correction stayed bounded.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Grounding targets met"] --> B["Next: trajectory evaluation"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---:|---:|
| Stale policy retrieval | Superseded policy ranked first | Current trusted revision selected |
| Grounded policy accuracy | Not measured | At least 90% on fixture suite |
| Citation coverage | Optional prose | Every accepted claim cited |
| Correction bound | Open-ended | At most 2 attempts |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Retrieval tool, metadata filter, relevance grade, rewrite, fallback, citation check |
| Explained and illustrated | Conflict handling, corrective and adaptive RAG |
| Named with a reason | Embedding training and vector indexes, already covered in GenAI retrieval chapters |

### Key Takeaways

- Retrieval similarity never grants authority.
- Filter by trust and revision before ranking relevance.
- Permit a small correction budget, then decline.
- A grounded answer is not complete until its citations validate.
